In [ ]:
route = [
'39133d1e-5672-4e93-ab3a-e22717f34e47.b_f32b8e47-d0a5-47a8-9ca0-a7b6c6229e82.a',
'0fc3d8e6-a1ba-4543-9858-1aaccdbe6375.a_f32b8e47-d0a5-47a8-9ca0-a7b6c6229e82.b',
'0fc3d8e6-a1ba-4543-9858-1aaccdbe6375.c_5e0d72a3-8a1d-4a1a-81df-7ae0bdd8e4df.a',
'373f61d3-9872-4231-9c4a-d2963920f8a1.a_5e0d72a3-8a1d-4a1a-81df-7ae0bdd8e4df.b'
]


In [ ]:
from models.Track import Track

In [16]:
for i in range(len(route) - 1):
    segment_end_ids = route[i].split("_") + route[i+1].split("_")
    segment_ids = [x.split(".")[0] for x in segment_end_ids]

    seen_segments = set()
    ends = {}
    for segment_end_id in segment_end_ids:
        if segment_end_id.split(".")[0] in seen_segments:
            common_segment_id = segment_end_id.split(".")[0]
            ends_ids = [segment_end_id.split(".")[1], ends[segment_end_id.split(".")[0]]]
        ends[segment_end_id.split(".")[0]] = segment_end_id.split(".")[1]
        seen_segments.add(segment_end_id.split(".")[0])
    print(common_segment_id, ends_ids)

f32b8e47-d0a5-47a8-9ca0-a7b6c6229e82 ['b', 'a']
0fc3d8e6-a1ba-4543-9858-1aaccdbe6375 ['c', 'a']
5e0d72a3-8a1d-4a1a-81df-7ae0bdd8e4df ['b', 'a']


In [ ]:
node_id = curr_node.split("_v")[0]

for neighbor in graph.neighbors(node_id):
    if neighbor == prev_node:
        if not neighbor in curr_virtual_nodes.get(node_id, []):
            continue

    curr_dists_since_switch_copy = curr_dists_since_switch.copy()
    curr_virtual_nodes_copy = curr_virtual_nodes.copy()
    print(curr_dists_since_switch_copy)

    # check if node_id is a key in curr_dists_since_switch_copy
    # print(node_id, curr_dists_since_switch_copy)
    # if node_id in curr_dists_since_switch_copy:
    #     print("found switch distance counter for node ", node_id)
    to_delete = []
    for key in curr_dists_since_switch_copy:
        curr_dists_since_switch_copy[key] += graph[node_id][neighbor].get('length', 0)
        if curr_dists_since_switch_copy[key] > train_length:
            to_delete.append(key)
        else:
            curr_virtual_nodes_copy[key].append(f"{node_id}_v")

    for key in to_delete:
        del curr_dists_since_switch_copy[key]
        for key, value in curr_virtual_nodes_copy.items():
            for node_key, node in enumerate(value):
                curr_virtual_nodes_copy[key][node_key] = node.split("_v")[0]

    if prev_node is not None and graph.degree(node_id) > 2:
        inbound_edge_type = graph[prev_node][node_id].get('type')
        outbound_edge_type = graph[node_id][neighbor].get('type')

        if inbound_edge_type == "diverging":
            curr_dists_since_switch_copy[node_id] = 0
            curr_virtual_nodes_copy[node_id] = []

        if inbound_edge_type == "diverging" and outbound_edge_type == "diverging":
            # curr_dists_since_switch_copy[curr_node] = 0
            # print(curr_dists_since_switch_copy)
            # print("Switching at node", node_id, "from", prev_node, "to", neighbor)
            continue

    edge_len = graph[node_id][neighbor].get('length', 0)
    new_total_dist = curr_total_dist + edge_len
    new_state = (neighbor, node_id, curr_dists_since_switch_copy, curr_virtual_nodes_copy, curr_path + [curr_node])
    print("new state:", new_state)

    if (neighbor, node_id) not in visited and ((neighbor, node_id) not in unvisited or new_total_dist < unvisited.get((neighbor, node_id), float('inf'))):
        unvisited[(neighbor, node_id)] = new_total_dist
        states[(neighbor, node_id)] = new_state
        previous_states[(neighbor, node_id)] = (curr_node, prev_node)
